# RAG with a persistent SQLite index

This notebook opens the FAQ index created during ingestion, checks lexical retrieval, and passes the same index to the reusable `RAGBase` assistant.

## 1. Open and inspect the persisted index

The index stores searchable text fields and a keyword field for exact course filtering. The database file must already exist; run the ingestion notebook first if `faq.db` has not been created.

In [ ]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    db_path='faq.db'
)

In [ ]:
sqlite_index.count()

In [ ]:
results = sqlite_index.search('How do I join the course?', num_results=10)
results

In [ ]:
[doc['question'] for doc in results]

## 2. Connect the SQLite index to the RAG assistant

`TextSearchIndex` exposes the same search interface expected by `RAGBase`, so the retrieval backend can change without rewriting the generation pipeline.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [ ]:
from rag_helper import RAGBase

instructions = """
You are a helpful assistant.
Answer the question based on the context provided. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
"""

assistant = RAGBase(
    index=sqlite_index, 
    llm_client=openai_client,
    instructions=instructions
)

In [ ]:
question = "I just discovered the course, can I still join?"

assistant.rag(question)

## 3. Inspect retrieval quality

Looking at the retrieved questions separately helps distinguish retrieval problems from answer-generation problems.

In [ ]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

In [ ]:
question = "How do I run Ollama locally?"

print(assistant.rag(question))

In [ ]:
sqlite_index.close()